# Team 7 — ESPnet VITS / LibriTTS data preparation and benchmark notebook

**Purpose:** validate the existing Kaldi-style `train/` manifests, understand which LibriTTS subsets are referenced, check audio paths/sample rates/durations.

References:
- ESPnet TTS recipe flow: https://espnet.github.io/espnet/recipe/tts1.html
- LibriTTS recipe: https://github.com/espnet/espnet/tree/master/egs2/libritts/tts1

## 0. Configure paths

In [2]:
from pathlib import Path

# TODO: Replace this with the actual absolute path to the directory containing:
# wav.scp, text, utt2spk, spk2utt, spk2gender
DATA_DIR = Path("/home/malavika/DATABASE/libritts_espnet_frmt/train")

MANIFESTS = ["wav.scp", "text", "utt2spk", "spk2utt", "spk2gender"]

print("Notebook working directory:", Path.cwd())
print("Data directory:", DATA_DIR)
print("Directory exists:", DATA_DIR.exists())

if DATA_DIR.exists():
    for name in MANIFESTS:
        p = DATA_DIR / name
        print(f"{name:12} exists={p.exists()}  bytes={p.stat().st_size if p.exists() else '—'}")

Notebook working directory: /home/grp7
Data directory: /home/malavika/DATABASE/libritts_espnet_frmt/train
Directory exists: True
wav.scp      exists=True  bytes=21036797
text         exists=True  bytes=18373962
utt2spk      exists=True  bytes=5463002
spk2utt      exists=True  bytes=3805762
spk2gender   exists=True  bytes=34074


## 1. Read and summarize the manifests

Kaldi-style files are generally whitespace-separated: the first field is an utterance or speaker ID and the rest is the value. For `text`, everything after the first whitespace is the transcript. For `wav.scp`, everything after the first whitespace is the audio source; it can be a path or, in some corpora, a command pipeline.

This cell reads the manifests without rewriting them.

In [3]:
def read_kaldi_map(path, value_can_contain_spaces=True):
    rows = []
    malformed = []
    with open(path, "r", encoding="utf-8") as f:
        for line_no, raw in enumerate(f, start=1):
            line = raw.rstrip("\\n")
            if not line.strip():
                continue
            parts = line.split(maxsplit=1)
            if len(parts) < 2:
                malformed.append((line_no, line))
                continue
            key, value = parts
            rows.append((key, value))
    return rows, malformed

if not DATA_DIR.exists():
    raise FileNotFoundError("Set DATA_DIR to the real manifest directory first.")

manifest_rows = {}
for name in MANIFESTS:
    p = DATA_DIR / name
    if p.exists():
        rows, malformed = read_kaldi_map(p)
        manifest_rows[name] = rows
        print(f"{name}: {len(rows):,} rows; malformed lines: {len(malformed)}")
        if malformed[:5]:
            print("  Examples:", malformed[:5])
    else:
        print(f"{name}: MISSING")

wav.scp: 149,329 rows; malformed lines: 0
text: 149,329 rows; malformed lines: 0
utt2spk: 149,329 rows; malformed lines: 0
spk2utt: 2,558 rows; malformed lines: 0
spk2gender: 2,558 rows; malformed lines: 0


## 2. Check duplicate IDs and cross-file consistency

The core invariant is that the utterance IDs in `wav.scp`, `text`, and `utt2spk` should match. `spk2utt` should be consistent with `utt2spk`. The speaker IDs in `spk2gender` should cover the speakers in `utt2spk` if gender metadata is expected for all speakers.

A missing item is a finding to investigate, not something to silently fill in.

In [4]:
from collections import Counter, defaultdict

def duplicate_keys(rows):
    counts = Counter(k for k, _ in rows)
    return [k for k, n in counts.items() if n > 1]

for name, rows in manifest_rows.items():
    dups = duplicate_keys(rows)
    print(f"{name}: duplicate keys = {len(dups)}")
    if dups[:10]:
        print("  Examples:", dups[:10])

def keyset(name):
    return {k for k, _ in manifest_rows.get(name, [])}

for name in ["text", "utt2spk"]:
    if "wav.scp" in manifest_rows and name in manifest_rows:
        wav_ids, other_ids = keyset("wav.scp"), keyset(name)
        print(f"wav.scp vs {name}: wav-only={len(wav_ids-other_ids):,}, {name}-only={len(other_ids-wav_ids):,}")

if "utt2spk" in manifest_rows and "spk2utt" in manifest_rows:
    utt2spk = dict(manifest_rows["utt2spk"])
    rebuilt = defaultdict(set)
    for utt, spk in manifest_rows["utt2spk"]:
        rebuilt[spk].add(utt)

    listed = {}
    for spk, utts_text in manifest_rows["spk2utt"]:
        listed[spk] = set(utts_text.split())

    all_speakers = set(rebuilt) | set(listed)
    mismatches = []
    for spk in all_speakers:
        if rebuilt.get(spk, set()) != listed.get(spk, set()):
            mismatches.append(spk)

    print("spk2utt / utt2spk speaker mapping mismatches:", len(mismatches))
    if mismatches[:10]:
        print("Examples:", mismatches[:10])

if "utt2spk" in manifest_rows and "spk2gender" in manifest_rows:
    utt_speakers = {spk for _, spk in manifest_rows["utt2spk"]}
    gender_speakers = {spk for spk, _ in manifest_rows["spk2gender"]}
    print("Speakers missing gender:", len(utt_speakers - gender_speakers))
    print("Gender entries for unused speakers:", len(gender_speakers - utt_speakers))

wav.scp: duplicate keys = 0
text: duplicate keys = 0
utt2spk: duplicate keys = 0
spk2utt: duplicate keys = 0
spk2gender: duplicate keys = 0
wav.scp vs text: wav-only=0, text-only=0
wav.scp vs utt2spk: wav-only=0, utt2spk-only=0
spk2utt / utt2spk speaker mapping mismatches: 5116
Examples: ['2853_164188', '125_121124', '7981_112058', '8468_295198\n', '1509_145742', '7520_258954', '4039_184264\n', '8008_271811\n', '8527_281132', '4806_26894\n']
Speakers missing gender: 2558
Gender entries for unused speakers: 2558


## 3. Inspect subset references in `wav.scp`

LibriTTS utterance IDs often encode speaker/chapter/utterance, while paths can reveal whether the audio comes from `train-clean-100` or `train-clean-360`. We inspect both IDs and path strings because directory layouts can vary.

This is a summary only; it does not assume that a folder name is always present.

In [ ]:
if "wav.scp" in manifest_rows:
    rows = manifest_rows["wav.scp"]
    markers = ["train-clean-100", "train-clean-360"] # "dev-clean", "test-clean"
    lower_rows = [(utt, src.lower()) for utt, src in rows]
    for marker in markers:
        hits = [(utt, src) for utt, src in lower_rows if marker in src]
        print(f"{marker:18}: {len(hits):,} manifest entries")
        for example in hits[:2]:
            print("   ", example)

train-clean-100   : 33,168 manifest entries
    ('1034_121119_000001_000001', '/home/malavika/database/libritts_espnet_frmt/data/libritts/train-clean-100/1034/121119/1034_121119_000001_000001.wav\n')
    ('1034_121119_000002_000001', '/home/malavika/database/libritts_espnet_frmt/data/libritts/train-clean-100/1034/121119/1034_121119_000002_000001.wav\n')
train-clean-360   : 116,161 manifest entries
    ('100_121669_000001_000000', '/home/malavika/database/libritts_espnet_frmt/data/libritts/train-clean-360/100/121669/100_121669_000001_000000.wav\n')
    ('100_121669_000003_000000', '/home/malavika/database/libritts_espnet_frmt/data/libritts/train-clean-360/100/121669/100_121669_000003_000000.wav\n')


## 4. Validate audio paths and inspect sample rates/durations

Install/use `soundfile` only if it is available in the notebook's Python environment. For ordinary LibriTTS entries where the value is a direct file path, this checks existence and reads metadata without loading all waveforms into RAM.

If `wav.scp` values contain pipes (`|`) or command strings, this basic path checker will flag them as non-direct paths; that is not automatically an error. ESPnet/Kaldi may support such entries, and the recipe should handle them appropriately.

In [8]:
try:
    import soundfile as sf
    HAVE_SOUNDFILE = True
except ImportError:
    HAVE_SOUNDFILE = False
    print("soundfile is not installed in this notebook environment.")

def looks_like_direct_path(src):
    s = src.strip()
    return bool(s) and "|" not in s and not s.endswith(")") and not s.startswith("sox ")

audio_report = {
    "direct_path": 0, "non_direct_entry": 0, "missing": 0,
    "read_error": 0, "checked": 0, "sample_rates": Counter(),
    "durations_sec": []
}
missing_examples, read_error_examples, non_direct_examples = [], [], []

for utt, src in manifest_rows.get("wav.scp", []):
    src = src.strip()
    if not looks_like_direct_path(src):
        audio_report["non_direct_entry"] += 1
        if len(non_direct_examples) < 10:
            non_direct_examples.append((utt, src))
        continue

    audio_path = Path(src)
    if not audio_path.is_absolute():
        audio_path = DATA_DIR / audio_path

    audio_report["direct_path"] += 1
    if not audio_path.is_file():
        audio_report["missing"] += 1
        if len(missing_examples) < 20:
            missing_examples.append((utt, str(audio_path)))
        continue

    if HAVE_SOUNDFILE:
        try:
            info = sf.info(str(audio_path))
            audio_report["checked"] += 1
            audio_report["sample_rates"][info.samplerate] += 1
            audio_report["durations_sec"].append(info.duration)
        except Exception as e:
            audio_report["read_error"] += 1
            if len(read_error_examples) < 10:
                read_error_examples.append((utt, str(audio_path), repr(e)))

print({k: v for k, v in audio_report.items() if k not in ["sample_rates", "durations_sec"]})
print("Sample-rate counts:", dict(audio_report["sample_rates"]))
print("Non-direct wav.scp examples:", non_direct_examples[:5])
print("Missing path examples:", missing_examples[:10])
print("Read error examples:", read_error_examples[:5])

if audio_report["durations_sec"]:
    import numpy as np
    d = np.asarray(audio_report["durations_sec"])
    print("Duration seconds: min / median / max =",
          round(float(d.min()), 3), round(float(np.median(d)), 3), round(float(d.max()), 3))
    print("Total duration of checked audio (hours):", round(float(d.sum()/3600), 2))

{'direct_path': 149329, 'non_direct_entry': 0, 'missing': 0, 'read_error': 0, 'checked': 149329}
Sample-rate counts: {24000: 149329}
Non-direct wav.scp examples: []
Missing path examples: []
Read error examples: []
Duration seconds: min / median / max = 0.16 4.58 34.53
Total duration of checked audio (hours): 243.96


## 5. Check transcript sanity

This catches empty transcripts, unusually short/long text, and unexpected control characters. It does not rewrite or normalize the text. English LibriTTS transcripts may contain punctuation, so punctuation itself should not be treated as a problem.

In [10]:
if "text" in manifest_rows:
    texts = [(utt, txt.strip()) for utt, txt in manifest_rows["text"]]

    empty = [(utt, txt) for utt, txt in texts if not txt]
    control = [
        (utt, txt) for utt, txt in texts
        if any(ord(c) < 32 and c not in "\t" for c in txt)]

    lengths = [len(txt) for _, txt in texts]

    print("Empty transcripts:", len(empty))
    print("Transcripts with unusual control chars:", len(control))

    if lengths:
        import numpy as np
        print("Transcript character length min/median/max:",
              min(lengths), float(np.median(lengths)), max(lengths))

    print("Examples:")
    for row in texts[:5]:
        print(row)

Empty transcripts: 0
Transcripts with unusual control chars: 0
Transcript character length min/median/max: 1 78.0 471
Examples:
('100_121669_000001_000000', "Tom, the Piper's Son")
('100_121669_000003_000000', "Tom, Tom, the piper's son, Stole a pig and away he run; The pig was eat and Tom was beat And Tom ran crying down the street.")
('100_121669_000004_000000', 'There was not a worse vagabond in Shrewsbury than old Barney the piper.')
('100_121669_000004_000001', 'He never did any work except to play the pipes, and he played so badly that few pennies ever found their way into his pouch.')
('100_121669_000004_000002', "It was whispered around that old Barney was not very honest, but he was so sly and cautious that no one had ever caught him in the act of stealing, although a good many things had been missed after they had fallen into the old man's way.")


## 6. Create a reproducible train/validation plan

Do **not** randomly split individual utterances into train and validation without thinking about speaker leakage. For TTS, it is common to use the recipe's official `dev-clean` validation set and `test-clean` evaluation set. The LibriTTS recipe defaults to `train-clean-460`, `dev-clean`, and test sets in its `run.sh`.

Because your current `train/` manifest combines `train-clean-100` and `train-clean-360`, it is important to verify whether separate official dev/test manifests already exist. Do not use evaluation transcripts/audio for training. For a quick benchmark, use the official training recipe's subset and validation flow if available rather than inventing a new split.

This cell gives a report of speaker and utterance counts; it does not split or copy data.

In [11]:
if "utt2spk" in manifest_rows:
    utt2spk_map = manifest_rows["utt2spk"]
    speakers = {spk for _, spk in utt2spk_map}
    print("Utterances in utt2spk:", len(utt2spk_map))
    print("Unique speakers:", len(speakers))
    counts = Counter(spk for _, spk in utt2spk_map)
    print("Utterances per speaker min / median / max:",
          min(counts.values()), sorted(counts.values())[len(counts)//2], max(counts.values()))

Utterances in utt2spk: 149329
Unique speakers: 2558
Utterances per speaker min / median / max: 1 48 486


## 7. What ESPnet data preparation will still do

Your manifests are only the input stage. Depending on the recipe flags and stages, ESPnet may still:
1. create or validate train/dev/test Kaldi-style directories;
2. reformat or dump audio into the selected format;
3. extract speaker embeddings if `--use_spk_embed true` is enabled;
4. filter utterances outside duration limits;
5. generate a token list using the selected text cleaner and G2P frontend;
6. collect TTS statistics;
7. train, decode, and evaluate.

The official LibriTTS recipe currently uses a 24 kHz configuration and Kaldi x-vectors by default, but VITS configurations may use different sample rates and feature settings. Inspect the exact VITS YAML and matching inference config before choosing flags. Do not simply combine the recipe's 24 kHz values with a 22.05 kHz VITS config.

## 8. Benchmark protocol: separate pipeline check from training-speed test

### A. Pipeline/data-preparation benchmark
Run only the data-preparation stages using the actual recipe's documented stage options. Inspect `run.sh` and `tts.sh` first; stage numbers can vary. Check that the command completes, expected directories are created, and logs contain no path, audio, tokenization, or speaker-embedding errors.

### B. Training benchmark
After data preparation succeeds:
- use the correct VITS training config and a modest, recipe-compatible batch size;
- set one GPU (`--ngpu 1`) for the RTX 4090;
- run a short controlled benchmark (for example, 500–2,000 steps if the recipe supports an iteration-limited run);
- record wall time, step time, peak VRAM, GPU utilization, and losses;
- save a checkpoint and test inference if enough training has occurred.

Do not blindly run the full `./run.sh` with default settings: the LibriTTS recipe defaults to four GPUs and a non-VITS `conf/train.yaml` in its standard invocation. We must confirm the correct VITS configuration and stage options first.

## 9. Optional: inspect the GPU from the notebook

This does not train anything.

In [ ]:
import subprocess

for cmd in [["nvidia-smi"], ["bash", "-lc", "python -c 'import torch; print(torch.__version__); print(torch.cuda.is_available()); print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no CUDA GPU")'"]]:
    print("$", " ".join(cmd))
    try:
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=30)
        print(result.stdout)
        if result.stderr:
            print("STDERR:", result.stderr[:2000])
        print("exit code:", result.returncode)
    except Exception as e:
        print("Could not run:", repr(e))

## 10. Benchmark record template

Fill this out after the short run. A benchmark is only useful if you record the exact config and dataset used.

In [ ]:
benchmark_record = {
    "espnet_git_commit": None,
    "recipe_path": None,
    "vits_config_path": None,
    "dataset_subset": "train-clean-100 subset / other: specify",
    "sample_rate_hz": None,
    "gpu": "NVIDIA RTX 4090 24 GB (verify on server)",
    "ngpu": 1,
    "batch_size_or_batch_bins": None,
    "num_steps": None,
    "wall_time_minutes": None,
    "seconds_per_step": None,
    "peak_vram_gb": None,
    "last_train_losses": None,
    "checkpoint_path": None,
    "inference_test_passed": None,
    "notes": None,
}
benchmark_record

## 11. Stop/go criteria

**Go forward** if:
- manifest IDs are consistent;
- required audio files are accessible;
- the selected sample rate/configuration matches the audio pipeline;
- text and speaker metadata pass checks;
- data preparation completes;
- the short run produces finite losses and a checkpoint.

**Stop and fix first** if:
- many audio paths are missing;
- IDs do not match between manifests;
- speaker mappings disagree;
- sample rate and model config are inconsistent;
- loss becomes NaN/Inf or training crashes;
- the output checkpoint cannot be loaded for inference.

### What to bring back from the server
1. Output from this notebook's manifest checks.
2. `nvidia-smi` and the PyTorch CUDA check.
3. The exact `run.sh` and VITS YAML filenames/configuration.
4. The command and log excerpt from the data-prep run.
5. Benchmark steps, elapsed time, peak VRAM, and whether inference worked.

With these, we can make a grounded two-day training decision instead of guessing.